# 🔬 Notebook 3: Amazon Lambda (Serverless) — Deep Dive

## 🛠️ Setup

```bash
cd 06-system-designs/amazon-lambda
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1

### Warm pool / container reuse

Cold start is ~500ms-1s (VM boot + runtime init). We keep a **pool of idle warm containers** per function for a TTL (e.g., 5 min of inactivity). Next invocation reuses one → 10ms start.

In [ ]:
import time
from collections import defaultdict, deque

class Pool:
    def __init__(self, ttl=5):
        self.pools = defaultdict(deque)     # function → deque[(container, idle_since)]
        self.ttl = ttl

    def acquire(self, fn):
        self._gc(fn)
        q = self.pools[fn]
        if q:
            c, _ = q.popleft()
            return c, "warm"
        return f"container-{fn}-{int(time.time()*1000)}", "cold"

    def release(self, fn, c):
        self.pools[fn].append((c, time.time()))

    def _gc(self, fn):
        now = time.time()
        q = self.pools[fn]
        while q and now - q[0][1] > self.ttl:
            q.popleft()

p = Pool()
print(p.acquire("hello"))      # cold
c, kind = p.acquire("hello")   # cold (first was handed out)
p.release("hello", c)
print(p.acquire("hello"))      # warm (reused)

## Deep dive 2

### Back-pressure & concurrency limits

Each function has a concurrency cap. Excess invocations queue (async) or 429 (sync). Below: a simple semaphore-limited executor.

In [ ]:
import threading, time, random

class Limiter:
    def __init__(self, cap): self.sem = threading.Semaphore(cap)
    def run(self, name, fn):
        if not self.sem.acquire(blocking=False):
            return "throttled-429"
        try:
            return fn()
        finally:
            self.sem.release()

lim = Limiter(cap=2)
def work(): time.sleep(0.01); return "ok"
results = []
for _ in range(5):
    results.append(lim.run("f", work))
print(results)   # some throttled

## Closing thoughts

- **Warm pools** are the main weapon against cold starts.
- Hard tenant isolation wants **microVMs** (Firecracker), not shared processes.
- Concurrency caps per function protect shared infra from a noisy tenant.